<a href="https://colab.research.google.com/github/GillValenzuela/curso_data_science/blob/master/DS_Ingemat_Clase_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.4 MB/s eta 0:00:00


In [ ]:
import torch, torch.nn as nn, torchvision
from torchvision import datasets, transforms
from tqdm import tqdm

In [ ]:
# Datos
tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_ds = datasets.MNIST(root="data", train=True,  download=True, transform=tfms)
val_ds   = datasets.MNIST(root="data", train=False, download=True, transform=tfms)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=256)


In [ ]:
# Red configurable
class DeepMLP(nn.Module):
    def __init__(self, hidden, p_drop=0.3, act=nn.ReLU):
        super().__init__()
        layers, in_dim = [], 28*28
        for h in hidden:
            layers += [nn.Linear(in_dim, h),
                       nn.BatchNorm1d(h),
                       act(),
                       nn.Dropout(p_drop)]
            in_dim = h
        layers += [nn.Linear(in_dim, 10)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

In [2]:
import matplotlib.pyplot as plt

In [ ]:
def train_model(hidden, p_drop, epochs=10, lr=1e-3):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = DeepMLP(hidden, p_drop).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit   = nn.CrossEntropyLoss()

    # 👇 Historias para graficar
    train_loss_hist, val_loss_hist, val_acc_hist = [], [], []

    for ep in range(epochs):
        # ---- Train ----
        model.train()
        running_loss = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            running_loss += loss.item() * xb.size(0)
        train_loss_hist.append(running_loss / len(train_dl.dataset))

        # ---- Val ----
        model.eval(); val_loss = 0; correct = total = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                logits = model(xb.to(device))
                loss   = crit(logits, yb.to(device))
                val_loss += loss.item() * xb.size(0)
                pred = logits.argmax(1).cpu()
                correct += (pred == yb).sum().item()
                total   += yb.size(0)
        val_loss_hist.append(val_loss / len(val_dl.dataset))
        val_acc = correct / total
        val_acc_hist.append(val_acc)

        print(f"Epoch {ep:02d} | train {train_loss_hist[-1]:.3f} "
              f"| val {val_loss_hist[-1]:.3f} | acc {val_acc:.4f}")

    # 👇 Graficar
    epochs_range = range(1, epochs+1)
    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.plot(epochs_range, train_loss_hist, label="train loss")
    plt.plot(epochs_range, val_loss_hist,   label="val loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Curva de pérdida")
    plt.legend(); plt.grid(True)

    plt.subplot(1,2,2)
    plt.plot(epochs_range, val_acc_hist)
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Val accuracy")
    plt.ylim(0.9,1.0); plt.grid(True)

    plt.tight_layout(); plt.show()